# 07b · Model Comparison with Class Imbalance Correction

**Project:** Enterprise HR AI  
**Task:** Train Logistic Regression (`class_weight='balanced'`), Random Forest (`class_weight='balanced'`), and XGBoost (`scale_pos_weight=5.2`) using the exact 80/20 stratified split (`random_state=42`).
Compare with Step 6 unweighted baseline. Select honest winner (Recall primary, F1 secondary), evaluate winner across multiple probability thresholds (0.3, 0.4, 0.5), and finalize production model.

---

In [1]:
import pandas as pd
import numpy as np
import os
import json
import joblib
import shutil
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.4f}'.format)

PROC = os.path.join('..', 'data', 'processed')
MODELS = os.path.join('..', 'models')
os.makedirs(MODELS, exist_ok=True)

RANDOM_STATE = 42
print('PROC  :', os.path.abspath(PROC))
print('MODELS:', os.path.abspath(MODELS))

PROC  : C:\Users\ASUS\Desktop\enterprise_hr_ai\data\processed
MODELS: C:\Users\ASUS\Desktop\enterprise_hr_ai\models


---
## Step 1 · Load Datasets and Reproduce 80/20 Stratified Split

Loading `features_scaled.csv` (for LogReg) and `features_unscaled.csv` (for tree models).

In [2]:
df_scaled = pd.read_csv(os.path.join(PROC, 'features_scaled.csv'))
df_unscaled = pd.read_csv(os.path.join(PROC, 'features_unscaled.csv'))

X_s = df_scaled.drop(columns=['Attrition'])
y_s = df_scaled['Attrition']

X_u = df_unscaled.drop(columns=['Attrition'])
y_u = df_unscaled['Attrition']

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_s, y_s, test_size=0.20, stratify=y_s, random_state=RANDOM_STATE
)

X_train_u, X_test_u, y_train_u, y_test_u = train_test_split(
    X_u, y_u, test_size=0.20, stratify=y_u, random_state=RANDOM_STATE
)

print(f'Train samples: {len(X_train_s)}, Test samples: {len(X_test_s)}')
print(f'Test leavers: {y_test_s.sum()} / {len(y_test_s)} ({y_test_s.mean()*100:.2f}%)')
assert (y_test_s.values == y_test_u.values).all(), 'Target alignment mismatch between scaled and unscaled splits!'

Train samples: 1176, Test samples: 294
Test leavers: 47 / 294 (15.99%)


---
## Step 2 · Train Class-Balanced Models

1. **Logistic Regression (Scaled)**: `class_weight='balanced'`, `max_iter=1000`
2. **Random Forest (Unscaled)**: `class_weight='balanced'`, `n_estimators=200`, `random_state=42`
3. **XGBoost (Unscaled)**: `scale_pos_weight=5.2`, `random_state=42`, `eval_metric='logloss'`

In [3]:
models = {}

print('1. Training Logistic Regression (class_weight="balanced")...')
lr_balanced = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)
lr_balanced.fit(X_train_s, y_train_s)
models['Logistic Regression (Balanced, Scaled)'] = (lr_balanced, X_test_s, y_test_s)

print('2. Training Random Forest (class_weight="balanced", n_estimators=200)...')
rf_balanced = RandomForestClassifier(class_weight='balanced', n_estimators=200, random_state=RANDOM_STATE)
rf_balanced.fit(X_train_u, y_train_u)
models['Random Forest (Balanced, Unscaled)'] = (rf_balanced, X_test_u, y_test_u)

print('3. Training XGBoost (scale_pos_weight=5.2)...')
xgb_balanced = XGBClassifier(scale_pos_weight=5.2, random_state=RANDOM_STATE, eval_metric='logloss', n_estimators=100)
xgb_balanced.fit(X_train_u, y_train_u)
models['XGBoost (scale_pos_weight=5.2, Unscaled)'] = (xgb_balanced, X_test_u, y_test_u)

print('All balanced models trained successfully.')

1. Training Logistic Regression (class_weight="balanced")...
2. Training Random Forest (class_weight="balanced", n_estimators=200)...


3. Training XGBoost (scale_pos_weight=5.2)...
All balanced models trained successfully.


---
## Step 3 · 4-Row Model Comparison Table

Including current production baseline (Step 6 unweighted LogReg).

In [4]:
table_rows = []

# Reference row: Step 6 Baseline
table_rows.append({
    'Model': 'Baseline LogReg (Step 6, Unweighted)',
    'Precision': 0.6538,
    'Recall': 0.3617,
    'F1': 0.4658,
    'ROC-AUC': 0.8134,
    'TN': 238, 'FP': 9, 'FN': 30, 'TP': 17
})

for name, (model, X_test, y_test) in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    cm = confusion_matrix(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)
    auc  = roc_auc_score(y_test, y_prob)
    
    table_rows.append({
        'Model': name,
        'Precision': prec,
        'Recall': rec,
        'F1': f1,
        'ROC-AUC': auc,
        'TN': cm[0, 0], 'FP': cm[0, 1], 'FN': cm[1, 0], 'TP': cm[1, 1]
    })

comp_df = pd.DataFrame(table_rows).set_index('Model')
print('=== 4-ROW BALANCED MODEL COMPARISON TABLE ===')
print(comp_df[['Precision', 'Recall', 'F1', 'ROC-AUC', 'TN', 'FP', 'FN', 'TP']].to_string())

=== 4-ROW BALANCED MODEL COMPARISON TABLE ===
                                          Precision  Recall     F1  ROC-AUC   TN  FP  FN  TP
Model                                                                                       
Baseline LogReg (Step 6, Unweighted)         0.6538  0.3617 0.4658   0.8134  238   9  30  17
Logistic Regression (Balanced, Scaled)       0.3810  0.6809 0.4885   0.8060  195  52  15  32
Random Forest (Balanced, Unscaled)           0.5000  0.0638 0.1132   0.7799  244   3  44   3
XGBoost (scale_pos_weight=5.2, Unscaled)     0.5517  0.3404 0.4211   0.7835  234  13  31  16


---
## Step 4 · Decision Rule & Winner Statement

Criteria: **Recall (Primary)**, **F1 (Secondary)**.

In [5]:
ranked = sorted(table_rows, key=lambda x: (x['Recall'], x['F1']), reverse=True)
print('=== MODELS RANKED STRICTLY BY (RECALL desc, F1 desc) ===')
for i, r in enumerate(ranked, 1):
    print(f"{i}. {r['Model']:42s} | Recall: {r['Recall']:.4f} | F1: {r['F1']:.4f} | Precision: {r['Precision']:.4f} | ROC-AUC: {r['ROC-AUC']:.4f} | TP: {r['TP']}/47")

winner = ranked[0]
print('\n' + '='*70)
print(f"HONEST WINNER: {winner['Model']}")
print('='*70)
print(f"Recall: {winner['Recall']:.4f} (caught {winner['TP']} of 47 leavers vs 17 in Step 6 baseline)")
print(f"F1: {winner['F1']:.4f}, Precision: {winner['Precision']:.4f}, ROC-AUC: {winner['ROC-AUC']:.4f}")

=== MODELS RANKED STRICTLY BY (RECALL desc, F1 desc) ===
1. Logistic Regression (Balanced, Scaled)     | Recall: 0.6809 | F1: 0.4885 | Precision: 0.3810 | ROC-AUC: 0.8060 | TP: 32/47
2. Baseline LogReg (Step 6, Unweighted)       | Recall: 0.3617 | F1: 0.4658 | Precision: 0.6538 | ROC-AUC: 0.8134 | TP: 17/47
3. XGBoost (scale_pos_weight=5.2, Unscaled)   | Recall: 0.3404 | F1: 0.4211 | Precision: 0.5517 | ROC-AUC: 0.7835 | TP: 16/47
4. Random Forest (Balanced, Unscaled)         | Recall: 0.0638 | F1: 0.1132 | Precision: 0.5000 | ROC-AUC: 0.7799 | TP: 3/47

HONEST WINNER: Logistic Regression (Balanced, Scaled)
Recall: 0.6809 (caught 32 of 47 leavers vs 17 in Step 6 baseline)
F1: 0.4885, Precision: 0.3810, ROC-AUC: 0.8060


---
## Step 5 · Multi-Threshold Evaluation for the Winning Candidate

Evaluating the winning model at probability thresholds: `[0.3, 0.4, 0.5]`.
Also save the candidate model to `models/attrition_pipeline_candidate.joblib` before finalization.

In [6]:
winner_name = winner['Model']
winner_tuple = models.get(winner_name)
best_model, best_X_test, best_y_test = winner_tuple

# Save initial candidate
cand_initial_path = os.path.join(MODELS, 'attrition_pipeline_candidate.joblib')
joblib.dump(best_model, cand_initial_path)
print(f'Initial candidate saved to: {cand_initial_path} ({os.path.getsize(cand_initial_path):,} bytes)')

probs = best_model.predict_proba(best_X_test)[:, 1]

thresh_rows = []
for t in [0.3, 0.4, 0.5]:
    preds_t = (probs >= t).astype(int)
    cm_t = confusion_matrix(best_y_test, preds_t)
    prec_t = precision_score(best_y_test, preds_t, zero_division=0)
    rec_t  = recall_score(best_y_test, preds_t, zero_division=0)
    f1_t   = f1_score(best_y_test, preds_t, zero_division=0)
    thresh_rows.append({
        'Threshold': t,
        'Precision': prec_t,
        'Recall': rec_t,
        'F1': f1_t,
        'TP (Caught)': cm_t[1, 1],
        'FN (Missed)': cm_t[1, 0],
        'FP (False Alarm)': cm_t[0, 1],
        'TN': cm_t[0, 0]
    })

thresh_df = pd.DataFrame(thresh_rows).set_index('Threshold')
print(f'=== MULTI-THRESHOLD PERFORMANCE FOR: {winner_name} ===')
print(thresh_df.to_string())

Initial candidate saved to: ..\models\attrition_pipeline_candidate.joblib (2,575 bytes)
=== MULTI-THRESHOLD PERFORMANCE FOR: Logistic Regression (Balanced, Scaled) ===
           Precision  Recall     F1  TP (Caught)  FN (Missed)  FP (False Alarm)   TN
Threshold                                                                           
0.3000        0.2794  0.8085 0.4153           38            9                98  149
0.4000        0.3426  0.7872 0.4774           37           10                71  176
0.5000        0.3810  0.6809 0.4885           32           15                52  195


---
## Step 6 · Finalize Production Model & Configuration

Finalizes production artifact, validates metrics dynamically, writes `models/model_config.json`, and archives the candidate.

In [7]:
import time

SELECTED_THRESHOLD = 0.40
print(f'1. Setting decision threshold to {SELECTED_THRESHOLD:.2f} on predicted probabilities.')

# Recompute metrics dynamically from confusion matrix at threshold=0.40
preds_040 = (probs >= SELECTED_THRESHOLD).astype(int)
cm_040 = confusion_matrix(best_y_test, preds_040)
tn, fp, fn, tp = cm_040.ravel()

computed_recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
computed_precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
computed_f1 = (2 * computed_precision * computed_recall / (computed_precision + computed_recall))

expected_recall = 0.7872
expected_precision = 0.3426
expected_f1 = 0.4774

print(f'   Computed directly from CM: Recall={computed_recall:.4f}, Precision={computed_precision:.4f}, F1={computed_f1:.4f}')
print(f'   Expected from report    : Recall={expected_recall:.4f}, Precision={expected_precision:.4f}, F1={expected_f1:.4f}')

diff_rec = abs(computed_recall - expected_recall)
diff_prec = abs(computed_precision - expected_precision)
diff_f1 = abs(computed_f1 - expected_f1)

if diff_rec < 0.0005 and diff_prec < 0.0005 and diff_f1 < 0.0005:
    print('   CONFIRMED: Computed metrics match reported metrics exactly (within rounding).')
else:
    print(f'   DISCREPANCY DETECTED! Delta: Recall={diff_rec:.5f}, Precision={diff_prec:.5f}, F1={diff_f1:.5f}')

# 2. Overwrite models/attrition_pipeline.joblib with visible before/after inspection
prod_path = os.path.join(MODELS, 'attrition_pipeline.joblib')
if os.path.exists(prod_path):
    old_size = os.path.getsize(prod_path)
    old_mtime = time.ctime(os.path.getmtime(prod_path))
    print(f'\n2. Current production file BEFORE overwrite:')
    print(f'   Path: {os.path.abspath(prod_path)}')
    print(f'   Size: {old_size:,} bytes | Last Modified: {old_mtime}')
else:
    print('\n2. No existing production file found before overwrite.')

# Sleep 1 second so timestamp change is distinctly visible
time.sleep(1.1)
joblib.dump(best_model, prod_path)
new_size = os.path.getsize(prod_path)
new_mtime = time.ctime(os.path.getmtime(prod_path))
print(f'   Production file AFTER overwrite:')
print(f'   Path: {os.path.abspath(prod_path)}')
print(f'   Size: {new_size:,} bytes | Last Modified: {new_mtime}')

# 3. Save models/model_config.json with exact required structure
today_str = datetime.now().strftime('%Y-%m-%d')
config_dict = {
    "model": "logistic_regression_balanced",
    "threshold": round(SELECTED_THRESHOLD, 2),
    "recall": round(computed_recall, 4),
    "precision": round(computed_precision, 4),
    "f1": round(computed_f1, 4),
    "trained_on": "features_scaled.csv",
    "date": today_str
}

config_path = os.path.join(MODELS, 'model_config.json')
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config_dict, f, indent=2)
print(f'\n3. Saved models/model_config.json with exact schema:')
print(json.dumps(config_dict, indent=2))

# 4. Move candidate file to archive
archive_dir = os.path.join(MODELS, 'archive')
os.makedirs(archive_dir, exist_ok=True)
cand_src = os.path.join(MODELS, 'attrition_pipeline_candidate.joblib')
cand_dst = os.path.join(archive_dir, 'attrition_pipeline_candidate_v1.joblib')

print(f'\n4. Moving candidate model to archive...')
if os.path.exists(cand_src):
    shutil.move(cand_src, cand_dst)

orig_exists = os.path.exists(cand_src)
arch_exists = os.path.exists(cand_dst)
print(f'   Original path ({cand_src}) exists: {orig_exists} (should be False)')
print(f'   Archive path  ({cand_dst}) exists: {arch_exists} (should be True)')
assert not orig_exists and arch_exists, 'Archive move verification failed!'
print('   CONFIRMED: Candidate file successfully moved to archive.')

# 5. Final confirmation block
print('\n' + '='*75)
print('FINAL PRODUCTION CONFIRMATION BLOCK')
print('='*75)
print(f'Model Type        : {config_dict["model"]} (LogisticRegression, class_weight=balanced)')
print(f'Training Features : {config_dict["trained_on"]}')
print(f'Decision Threshold: {config_dict["threshold"]:.2f}')
print(f'Recall            : {config_dict["recall"]:.4f}  (TP={tp}/47 leavers caught, FN={fn} missed)')
print(f'Precision         : {config_dict["precision"]:.4f}  (FP={fp} false alarms, TN={tn})')
print(f'F1 Score          : {config_dict["f1"]:.4f}')
print(f'Production Model  : {os.path.abspath(prod_path)}')
print(f'Config File       : {os.path.abspath(config_path)}')
print(f'Archived File     : {os.path.abspath(cand_dst)}')
print('='*75)

1. Setting decision threshold to 0.40 on predicted probabilities.
   Computed directly from CM: Recall=0.7872, Precision=0.3426, F1=0.4774
   Expected from report    : Recall=0.7872, Precision=0.3426, F1=0.4774
   CONFIRMED: Computed metrics match reported metrics exactly (within rounding).

2. Current production file BEFORE overwrite:
   Path: C:\Users\ASUS\Desktop\enterprise_hr_ai\models\attrition_pipeline.joblib
   Size: 2,575 bytes | Last Modified: Tue Sep  1 03:09:17 2026


   Production file AFTER overwrite:
   Path: C:\Users\ASUS\Desktop\enterprise_hr_ai\models\attrition_pipeline.joblib
   Size: 2,575 bytes | Last Modified: Tue Sep  1 03:14:13 2026

3. Saved models/model_config.json with exact schema:
{
  "model": "logistic_regression_balanced",
  "threshold": 0.4,
  "recall": 0.7872,
  "precision": 0.3426,
  "f1": 0.4774,
  "trained_on": "features_scaled.csv",
  "date": "2026-09-01"
}

4. Moving candidate model to archive...
   Original path (..\models\attrition_pipeline_candidate.joblib) exists: False (should be False)
   Archive path  (..\models\archive\attrition_pipeline_candidate_v1.joblib) exists: True (should be True)
   CONFIRMED: Candidate file successfully moved to archive.

FINAL PRODUCTION CONFIRMATION BLOCK
Model Type        : logistic_regression_balanced (LogisticRegression, class_weight=balanced)
Training Features : features_scaled.csv
Decision Threshold: 0.40
Recall            : 0.7872  (TP=37/47 leavers caught, FN=10 missed)
Precision  